In [5]:
"""
Functional time series simulation: FPCA vs FAE
- Linear scenario: close to original spirit
- Nonlinear scenario: phase-warped fertility-like curves, intentionally favorable to FAE

Outputs:
- reconstruction relMSE vs CLEAN / NOISY
- forecast relMSE vs CLEAN / NOISY
- batch runner to Excel

Dependencies:
    numpy
    pandas
    torch
    openpyxl
"""

from __future__ import annotations

import copy
import math
import random
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch import optim


# ============================================================
# 0) Repro / metrics / utilities
# ============================================================

def set_seed(seed: int = 123) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def rel_mse(y_hat: torch.Tensor, y_true: torch.Tensor, eps: float = 1e-12) -> float:
    mse = torch.mean((y_hat - y_true) ** 2)
    var = torch.var(y_true, unbiased=False)
    return float((mse / var.clamp_min(eps)).detach().cpu())


def mse(a: torch.Tensor, b: torch.Tensor) -> float:
    return float(torch.mean((a - b) ** 2).detach().cpu())


def chrono_split(X: torch.Tensor, horizon: int) -> Tuple[torch.Tensor, torch.Tensor]:
    if horizon < 1:
        raise ValueError("horizon must be >= 1")
    if X.shape[0] <= horizon:
        raise ValueError("Need more observations than horizon.")
    return X[:-horizon], X[-horizon:]


def trapezoid_weights(x: np.ndarray) -> torch.Tensor:
    x = np.asarray(x, dtype=float)
    dx = np.diff(x)
    if len(dx) < 1:
        raise ValueError("Need at least 2 points.")
    w = np.zeros_like(x)
    w[0] = dx[0] / 2.0
    w[-1] = dx[-1] / 2.0
    if len(x) > 2:
        w[1:-1] = (x[2:] - x[:-2]) / 2.0
    return torch.tensor(w, dtype=torch.float32)


def print_report_table(title: str, rows: Dict[str, Dict[str, float]]) -> None:
    print("\n" + "=" * len(title))
    print(title)
    print("=" * len(title))
    header = f"{'Method':<16}  {'relMSE vs CLEAN':>16}  {'relMSE vs NOISY':>16}"
    print(header)
    print("-" * len(header))
    for method, vals in rows.items():
        a = vals.get("relMSE_vs_clean", float("nan"))
        b = vals.get("relMSE_vs_noisy", float("nan"))
        print(f"{method:<16}  {a:16.6f}  {b:16.6f}")


# ============================================================
# 1) Basis builder
# ============================================================

class BasisFCBuilder:
    def __init__(self, n_basis=20, basis_type="Bspline", bspline_degree=3):
        self.n_basis = int(n_basis)
        self.basis_type_l = basis_type.lower()
        self.bspline_degree = int(bspline_degree)

    def build(self, tpts: torch.Tensor) -> torch.Tensor:
        if self.basis_type_l == "fourier":
            return self._build_fourier(tpts)
        elif self.basis_type_l in ("bspline", "b-spline", "b_spline"):
            return self._build_bspline(tpts, self.bspline_degree)
        else:
            raise ValueError("basis_type must be 'Fourier' or 'Bspline'.")

    def _build_fourier(self, tpts: torch.Tensor) -> torch.Tensor:
        t = tpts.flatten()
        t_min = t.min()
        t_max = t.max()
        denom = (t_max - t_min).clamp_min(1e-8)
        tau = (t - t_min) / denom

        n_time = t.shape[0]
        n_basis = self.n_basis
        B = torch.zeros(n_time, n_basis, dtype=torch.float32, device=t.device)
        if n_basis > 0:
            B[:, 0] = 1.0

        k = 1
        idx = 1
        while idx < n_basis:
            B[:, idx] = torch.sin(2.0 * math.pi * k * tau)
            idx += 1
            if idx < n_basis:
                B[:, idx] = torch.cos(2.0 * math.pi * k * tau)
                idx += 1
            k += 1
        return B

    def _build_bspline(self, tpts: torch.Tensor, degree: int) -> torch.Tensor:
        t = tpts.flatten()
        t_min = t.min()
        t_max = t.max()
        denom = (t_max - t_min).clamp_min(1e-8)
        tau = (t - t_min) / denom
        tau_np = tau.detach().cpu().numpy()

        n_time = tau_np.shape[0]
        n_basis = self.n_basis
        p = degree

        if n_basis < p + 1:
            raise ValueError(f"n_basis must be >= degree+1={p+1}")

        n_int = max(n_basis - p - 1, 0)
        if n_int > 0:
            interior = np.linspace(0.0, 1.0, n_int + 2)[1:-1]
            knots = np.concatenate((np.zeros(p + 1), interior, np.ones(p + 1)))
        else:
            knots = np.concatenate((np.zeros(p + 1), np.ones(p + 1)))

        N = np.zeros((n_basis, n_time), dtype=np.float64)
        for i in range(n_basis):
            left = knots[i]
            right = knots[i + 1]
            N[i, :] = np.where((tau_np >= left) & (tau_np < right), 1.0, 0.0)
        N[-1, tau_np == 1.0] = 1.0

        for k in range(1, p + 1):
            N_next = np.zeros_like(N)
            for i in range(n_basis):
                denom_left = knots[i + k] - knots[i]
                if denom_left > 0:
                    c_left = (tau_np - knots[i]) / denom_left
                    left = c_left * N[i, :]
                else:
                    left = 0.0

                denom_right = (knots[i + k + 1] - knots[i + 1]) if (i + 1) < n_basis else 0.0
                if denom_right > 0 and (i + 1) < n_basis:
                    c_right = (knots[i + k + 1] - tau_np) / denom_right
                    right = c_right * N[i + 1, :]
                else:
                    right = 0.0

                N_next[i, :] = left + right
            N = N_next

        return torch.tensor(N.T, dtype=torch.float32, device=tpts.device)


# ============================================================
# 2) Latent processes
# ============================================================

def spectral_radius(A: np.ndarray) -> float:
    return float(np.max(np.abs(np.linalg.eigvals(A))))


def stabilize_A(A: np.ndarray, target_rho: float = 0.9) -> np.ndarray:
    rho = spectral_radius(A)
    if rho <= target_rho or rho <= 1e-12:
        return A
    return (target_rho / rho) * A


def generate_latent_var1(
    T: int,
    d: int,
    Sigma: np.ndarray,
    seed: int,
    target_rho: float = 0.80,
) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    A = rng.normal(size=(d, d)) * 0.15
    A = stabilize_A(A, target_rho=target_rho)

    Z = np.zeros((T, d), dtype=float)
    Z[0] = rng.multivariate_normal(np.zeros(d), Sigma)
    for t in range(1, T):
        eps = rng.multivariate_normal(np.zeros(d), Sigma)
        Z[t] = A @ Z[t - 1] + eps
    return Z, A


def generate_latent_phase_friendly(
    T: int,
    d: int,
    Sigma: np.ndarray,
    seed: int,
    clip_value: float = 2.0,
) -> np.ndarray:
    """
    Nonlinear latent process for the nonlinear scenario.

    Designed so that:
    - latent state remains stable
    - one coordinate drives phase shift over time
    - nonlinear but bounded dynamics
    """
    rng = np.random.default_rng(seed)
    Z = np.zeros((T, d), dtype=float)
    Z[0] = rng.multivariate_normal(np.zeros(d), Sigma)

    A = np.array([
        [0.60, 0.05, 0.00, 0.00, 0.00],
        [0.00, 0.72, 0.06, 0.00, 0.00],
        [0.00, 0.00, 0.66, 0.05, 0.00],
        [0.00, 0.00, 0.00, 0.68, 0.06],
        [0.03, 0.00, 0.00, 0.00, 0.62],
    ], dtype=float)[:d, :d]

    for t in range(1, T):
        eps = rng.multivariate_normal(np.zeros(d), Sigma)
        z = Z[t - 1].copy()
        tau = t / max(T - 1, 1)

        drift = np.zeros(d)
        drift[0] = -0.25 * (tau - 0.5)       # mild level trend
        if d >= 2:
            drift[1] = 1.10 * (tau - 0.5)    # strong postponement / phase trend
        if d >= 3:
            drift[2] = 0.25 * np.sin(2 * np.pi * tau)
        if d >= 4:
            drift[3] = 0.20 * np.sin(np.pi * tau)
        if d >= 5:
            drift[4] = 0.15 * np.cos(2 * np.pi * tau)

        nonlin = np.zeros(d)
        if d >= 1:
            z1 = z[1] if d >= 2 else z[0]
            nonlin[0] = 0.10 * np.tanh(1.0 * z1) - 0.04 * np.tanh(0.8 * z[0] * z1)
        if d >= 2:
            z2 = z[2] if d >= 3 else z[1]
            nonlin[1] = 0.18 * np.tanh(1.0 * z2) + 0.04 * np.tanh(z[0] ** 2)
        if d >= 3:
            z3 = z[3] if d >= 4 else z[2]
            nonlin[2] = -0.10 * np.tanh(1.0 * z[0]) + 0.08 * np.tanh(0.8 * z3)
        if d >= 4:
            nonlin[3] = 0.08 * np.sin(np.clip(z[1], -3.0, 3.0)) + 0.04 * np.tanh(0.6 * z[2] * z[3])
        if d >= 5:
            nonlin[4] = 0.06 * np.tanh(0.8 * z[4]) + 0.03 * np.tanh(0.6 * z[1] * z[2])

        znew = A @ z + nonlin + 0.05 * drift + eps
        znew = np.clip(znew, -clip_value, clip_value)
        Z[t] = znew

    return Z


# ============================================================
# 3) Curve generators
# ============================================================

class LinearMap(nn.Module):
    def __init__(self, d: int, hidden: List[int], M: int, bias: bool = True, activation=nn.Identity()):
        super().__init__()
        dims = [d] + hidden + [M]
        self.layers = nn.ModuleList(
            [nn.Linear(dims[i], dims[i + 1], bias=bias) for i in range(len(dims) - 1)]
        )
        self.activation = activation

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x = z
        for layer in self.layers:
            x = self.activation(layer(x))
        return x


def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    return torch.sigmoid(x)


def phase_warped_fertility_decoder(Z: torch.Tensor, cfg: "SimCfg") -> torch.Tensor:
    """
    Nonlinear curve generator with strong phase variation.
    This is the key change that tends to make FAE better than FPCA.
    """
    T = Z.shape[0]
    device = Z.device

    ages = torch.linspace(cfg.age_min, cfg.age_max, cfg.P, device=device, dtype=torch.float32)
    u = (ages - cfg.age_min) / (cfg.age_max - cfg.age_min)
    u = u.view(1, -1)

    z1 = Z[:, 0]
    z2 = Z[:, 1] if Z.shape[1] >= 2 else torch.zeros_like(z1)
    z3 = Z[:, 2] if Z.shape[1] >= 3 else torch.zeros_like(z1)
    z4 = Z[:, 3] if Z.shape[1] >= 4 else torch.zeros_like(z1)
    z5 = Z[:, 4] if Z.shape[1] >= 5 else torch.zeros_like(z1)

    trend = torch.linspace(-1.0, 1.0, T, device=device)
    z1t = z1 - 0.20 * trend
    z2t = z2 + 1.00 * trend
    z3t = z3
    z4t = z4
    z5t = z5

    # amplitude and shape controls
    amp = 0.07 + 0.07 * sigmoid_torch(1.0 * z1t - 0.6 * z2t)
    shoulder_amp = 0.16 * sigmoid_torch(1.1 * z4t + 0.5 * z5t)
    tail_amp = cfg.fertility_tail_strength * sigmoid_torch(1.1 * z5t + 0.5 * z4t)

    # phase / tempo shift
    shift = 0.14 * torch.tanh(1.4 * z2t - 0.7 * z3t + 0.3 * z1t * z4t)
    dilation = 1.0 + 0.18 * torch.tanh(1.1 * z3t + 0.3 * z5t)

    shift = shift.view(T, 1)
    dilation = dilation.view(T, 1)

    local_warp = (
        0.08
        * torch.tanh(1.3 * z4t - 0.6 * z2t + 0.2 * z1t * z5t).view(T, 1)
        * torch.exp(-0.5 * ((u - 0.48) / 0.16) ** 2)
        * torch.sin(math.pi * (u - 0.15))
    )

    uw = (u - shift + local_warp)
    uw = 0.50 + (uw - 0.50) / dilation
    uw = uw.clamp(-0.2, 1.2)

    # base fertility-like template
    main_peak = torch.exp(-0.5 * ((uw - 0.38) / 0.08) ** 2)
    shoulder = torch.exp(-0.5 * ((uw - 0.58) / 0.12) ** 2)
    tail = sigmoid_torch((uw - 0.62) / 0.04) * torch.exp(-(uw - 0.62).clamp_min(0.0) / 0.13)

    X = amp.view(T, 1) * main_peak
    X = X + shoulder_amp.view(T, 1) * amp.view(T, 1) * shoulder
    X = X + tail_amp.view(T, 1) * tail
    X = X + cfg.fertility_floor

    return X.clamp_min(0.0)


# ============================================================
# 4) Simulation config + simulator
# ============================================================

@dataclass
class SimCfg:
    seed: int = 123
    T: int = 500
    P: int = 100
    d: int = 5
    Sigma_scale: float = 0.03
    target_rho: float = 0.85

    gen_basis_type: str = "Bspline"
    gen_n_basis: int = 12
    gen_bspline_degree: int = 3

    map_mode: str = "nonlinear"   # "linear" or "nonlinear"
    map_weight_sd: float = 0.8
    meas_noise_sd: float = 0.0012

    age_min: float = 15.0
    age_max: float = 49.0
    fertility_tail_strength: float = 0.010
    fertility_floor: float = 0.0
    fertility_scale: float = 1.0


def simulate_functional_ts(cfg: SimCfg) -> Dict[str, object]:
    set_seed(cfg.seed)

    if cfg.map_mode.lower() == "nonlinear":
        u = np.linspace(cfg.age_min, cfg.age_max, cfg.P).astype(float)
    else:
        u = np.linspace(0.0, 1.0, cfg.P).astype(float)

    tpts = torch.tensor(u, dtype=torch.float32)
    Sigma = (cfg.Sigma_scale ** 2) * np.eye(cfg.d)

    if cfg.map_mode.lower() == "linear":
        Z_np, A_true = generate_latent_var1(
            T=cfg.T, d=cfg.d, Sigma=Sigma, seed=cfg.seed, target_rho=cfg.target_rho
        )
    else:
        Z_np = generate_latent_phase_friendly(
            T=cfg.T, d=cfg.d, Sigma=Sigma, seed=cfg.seed
        )
        A_true = None

    Z = torch.tensor(Z_np, dtype=torch.float32)
    if not torch.isfinite(Z).all():
        raise ValueError("Latent process produced non-finite values.")

    if cfg.map_mode.lower() == "linear":
        gen_builder = BasisFCBuilder(
            n_basis=cfg.gen_n_basis,
            basis_type=cfg.gen_basis_type,
            bspline_degree=cfg.gen_bspline_degree,
        )
        Bgen = gen_builder.build(tpts)

        mapper = LinearMap(d=cfg.d, hidden=[20], M=cfg.gen_n_basis, bias=True)
        for m in mapper.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=cfg.map_weight_sd)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        mapper.eval()
        with torch.no_grad():
            coef = mapper(Z)
            X_clean = coef @ Bgen.T
            X_noisy = X_clean + cfg.meas_noise_sd * torch.randn_like(X_clean)
    else:
        with torch.no_grad():
            X_clean = phase_warped_fertility_decoder(Z, cfg)
            X_noisy = X_clean + cfg.meas_noise_sd * torch.randn_like(X_clean)

    if not torch.isfinite(X_clean).all() or not torch.isfinite(X_noisy).all():
        raise ValueError("Simulated curves contain non-finite values.")

    return {
        "u": u,
        "tpts": tpts,
        "X_clean": X_clean,
        "X_noisy": X_noisy,
        "A_true": A_true,
    }


# ============================================================
# 5) Centered FPCA
# ============================================================

@torch.no_grad()
def centered_weighted_fpca_fit(X_train: torch.Tensor, w: torch.Tensor, K: int):
    mu = X_train.mean(dim=0)
    Xc = X_train - mu.unsqueeze(0)

    sw = torch.sqrt(w).view(1, -1)
    Xw = Xc * sw

    U, S, Vh = torch.linalg.svd(Xw, full_matrices=False)
    V = Vh.transpose(0, 1)

    K = min(K, V.shape[1])
    phi = (V[:, :K] / sw.flatten()[:, None]).T

    for k in range(K):
        nrm = torch.sqrt(torch.sum(phi[k] * phi[k] * w))
        phi[k] = phi[k] / nrm.clamp_min(1e-12)

    scores = (Xc * w.view(1, -1)) @ phi.T
    return mu, phi, scores


@torch.no_grad()
def fpca_reconstruct_from_trainfit(X_train: torch.Tensor, X_eval: torch.Tensor, u: np.ndarray, K: int):
    w = trapezoid_weights(u)
    mu, phi, scores_train = centered_weighted_fpca_fit(X_train, w, K)
    scores_eval = ((X_eval - mu.unsqueeze(0)) * w.view(1, -1)) @ phi.T
    Xhat_eval = mu.unsqueeze(0) + scores_eval @ phi
    return Xhat_eval, mu, phi, scores_train


# ============================================================
# 6) VAR helpers
# ============================================================

def fit_var1(H: np.ndarray, ridge: float = 1e-6):
    T, K = H.shape
    if T < 2:
        raise ValueError("Need at least 2 time points for VAR(1).")
    X = H[:-1, :]
    Y = H[1:, :]
    X_aug = np.hstack([np.ones((T - 1, 1)), X])
    XtX = X_aug.T @ X_aug
    B = np.linalg.solve(XtX + ridge * np.eye(XtX.shape[0]), X_aug.T @ Y)
    b = B[0, :]
    A = B[1:, :].T
    return b, A


def forecast_var1(h_last: np.ndarray, b: np.ndarray, A: np.ndarray, steps: int) -> np.ndarray:
    K = h_last.shape[0]
    out = np.zeros((steps, K), dtype=float)
    h = h_last.copy()
    for i in range(steps):
        h = b + (A @ h)
        out[i, :] = h
    return out


@torch.no_grad()
def fpca_var_forecast(X_train: torch.Tensor, u: np.ndarray, K: int, steps: int, var_ridge: float = 1e-6):
    w = trapezoid_weights(u)
    mu, phi, scores = centered_weighted_fpca_fit(X_train, w, K)
    H = scores.detach().cpu().numpy()
    b, A = fit_var1(H, ridge=var_ridge)
    Hf = forecast_var1(H[-1], b, A, steps=steps)
    Hf_t = torch.tensor(Hf, dtype=torch.float32)
    X_fore = mu.unsqueeze(0) + Hf_t @ phi
    return X_fore


# ============================================================
# 7) FAE with dynamics penalty
# ============================================================

class FAEVanilla(nn.Module):
    def __init__(self, n_basis_project: int, n_rep: int, n_basis_revert: int, init_weight_sd: Optional[float] = None):
        super().__init__()
        self.fc1 = nn.Linear(n_basis_project, 128, bias=True)
        self.fc2 = nn.Linear(128, n_rep, bias=True)
        self.fc3 = nn.Linear(n_rep, 128, bias=True)
        self.fc4 = nn.Linear(128, n_basis_revert, bias=True)
        self.activation = nn.Tanh()

        if init_weight_sd is not None:
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.normal_(m.weight, mean=0.0, std=init_weight_sd)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

    def project(self, x: torch.Tensor, tpts: torch.Tensor, basis_fc: torch.Tensor) -> torch.Tensor:
        t = tpts.flatten()
        dt = t[1:] - t[:-1]
        zero = torch.zeros(1, device=x.device, dtype=x.dtype)
        W = 0.5 * torch.cat([zero, dt]) + 0.5 * torch.cat([dt, zero])

        n_time = x.shape[1]
        if basis_fc.shape[0] == n_time:
            B = basis_fc
        elif basis_fc.shape[1] == n_time:
            B = basis_fc.T
        else:
            raise RuntimeError(f"basis_fc shape {tuple(basis_fc.shape)} incompatible with n_time={n_time}")

        return (x * W) @ B

    def revert(self, coef: torch.Tensor, basis_fc: torch.Tensor) -> torch.Tensor:
        n_basis = coef.shape[1]
        if basis_fc.shape[1] == n_basis:
            return coef @ basis_fc.T
        elif basis_fc.shape[0] == n_basis:
            return coef @ basis_fc
        else:
            raise RuntimeError(f"basis_fc shape {tuple(basis_fc.shape)} incompatible with n_basis={n_basis}")

    def decode_from_rep(self, rep: torch.Tensor, basis_fc_revert: torch.Tensor) -> torch.Tensor:
        h = self.activation(self.fc3(rep))
        coef = self.fc4(h)
        return self.revert(coef, basis_fc_revert)

    def forward(self, x: torch.Tensor, tpts: torch.Tensor, basis_fc_project: torch.Tensor, basis_fc_revert: torch.Tensor):
        feat = self.project(x, tpts, basis_fc_project)
        h = self.activation(self.fc1(feat))
        rep = self.fc2(h)
        h2 = self.activation(self.fc3(rep))
        coef = self.fc4(h2)
        x_hat = self.revert(coef, basis_fc_revert)
        return x_hat, rep, coef


def diff_penalty(coef: torch.Tensor) -> torch.Tensor:
    delta = coef[:, 2:] - 2 * coef[:, 1:-1] + coef[:, :-2]
    return torch.mean(torch.sum(delta ** 2, dim=1))


def latent_var_penalty(H: torch.Tensor, ridge: float = 1e-6) -> torch.Tensor:
    """
    Penalize one-step latent VAR residuals on a chronological block.
    """
    T, r = H.shape
    if T < 3:
        return torch.tensor(0.0, device=H.device, dtype=H.dtype)

    X = H[:-1, :]
    Y = H[1:, :]
    ones = torch.ones((X.shape[0], 1), device=H.device, dtype=H.dtype)
    X_aug = torch.cat([ones, X], dim=1)

    XtX = X_aug.T @ X_aug
    I = torch.eye(XtX.shape[0], device=H.device, dtype=H.dtype)
    B = torch.linalg.solve(XtX + ridge * I, X_aug.T @ Y)

    Y_hat = X_aug @ B
    return torch.mean((Y - Y_hat) ** 2)


@dataclass
class FaeCfg:
    seed: int = 743
    device: str = "cpu"

    n_basis_project: int = 45
    n_basis_revert: int = 45
    basis_type_project: str = "Bspline"
    basis_type_revert: str = "Bspline"
    bspline_degree: int = 3

    n_rep: int = 3
    init_weight_sd: float = 0.06

    epochs: int = 1400
    batch_size: int = 16
    lr: float = 7e-4
    weight_decay: float = 1e-5
    split_rate: float = 0.85
    log_every: int = 200

    smooth_lambda: float = 2e-4
    dyn_lambda: float = 8.0
    var_ridge: float = 1e-6


def train_fae_on_train(Xn_train: torch.Tensor, Xc_train: torch.Tensor, tpts: torch.Tensor, cfg: FaeCfg):
    set_seed(cfg.seed)
    device = torch.device(cfg.device)

    proj_builder = BasisFCBuilder(
        n_basis=cfg.n_basis_project,
        basis_type=cfg.basis_type_project,
        bspline_degree=cfg.bspline_degree,
    )
    rev_builder = BasisFCBuilder(
        n_basis=cfg.n_basis_revert,
        basis_type=cfg.basis_type_revert,
        bspline_degree=cfg.bspline_degree,
    )

    Bp = proj_builder.build(tpts.to(device)).to(device)
    Br = rev_builder.build(tpts.to(device)).to(device)
    tpts_d = tpts.to(device).float()

    loss_fn = nn.MSELoss()

    n = Xn_train.shape[0]
    n_tr = max(8, int(cfg.split_rate * n))
    n_tr = min(n_tr, n - 1)

    Xn_tr = Xn_train[:n_tr].float().to(device)
    Xc_tr = Xc_train[:n_tr].float().to(device)
    Xn_va = Xn_train[n_tr:].float().to(device)
    Xc_va = Xc_train[n_tr:].float().to(device)

    model = FAEVanilla(
        cfg.n_basis_project, cfg.n_rep, cfg.n_basis_revert, init_weight_sd=cfg.init_weight_sd
    ).to(device)

    opt = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_state = None
    best_val = float("inf")

    for ep in range(1, cfg.epochs + 1):
        model.train()

        # minibatch reconstruction step
        for start in range(0, Xn_tr.shape[0], cfg.batch_size):
            xb_in = Xn_tr[start:start + cfg.batch_size]
            xb_tg = Xc_tr[start:start + cfg.batch_size]

            opt.zero_grad()
            xhat, rep, coef = model(xb_in, tpts_d, Bp, Br)
            loss = loss_fn(xhat, xb_tg) + cfg.smooth_lambda * diff_penalty(coef)
            loss.backward()
            opt.step()

        # full chronological dynamics step
        model.train()
        opt.zero_grad()
        xhat_full, H_full, coef_full = model(Xn_tr, tpts_d, Bp, Br)
        dyn_loss = latent_var_penalty(H_full, ridge=cfg.var_ridge)
        recon_full = loss_fn(xhat_full, Xc_tr)
        smooth_full = diff_penalty(coef_full)
        full_loss = recon_full + cfg.smooth_lambda * smooth_full + cfg.dyn_lambda * dyn_loss
        full_loss.backward()
        opt.step()

        if cfg.log_every and (ep % cfg.log_every == 0):
            model.eval()
            with torch.no_grad():
                tr_hat, H_tr, coef_tr = model(Xn_tr, tpts_d, Bp, Br)
                va_hat, H_va, coef_va = model(Xn_va, tpts_d, Bp, Br)

                tr_recon = loss_fn(tr_hat, Xc_tr)
                va_recon = loss_fn(va_hat, Xc_va)
                tr_dyn = latent_var_penalty(H_tr, ridge=cfg.var_ridge)
                va_dyn = latent_var_penalty(H_va, ridge=cfg.var_ridge)

                tr_obj = tr_recon + cfg.smooth_lambda * diff_penalty(coef_tr) + cfg.dyn_lambda * tr_dyn
                va_obj = va_recon + cfg.smooth_lambda * diff_penalty(coef_va) + cfg.dyn_lambda * va_dyn

            print(
                f"[FAE] ep {ep:4d} | train_mse(clean_target)={float(tr_recon):.6e} "
                f"| val_mse(clean_target)={float(va_recon):.6e}"
            )

            if float(va_obj) < best_val:
                best_val = float(va_obj)
                best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, tpts_d.detach().cpu(), Bp.detach().cpu(), Br.detach().cpu()


@torch.no_grad()
def fae_reconstruct(model: FAEVanilla, X: torch.Tensor, tpts: torch.Tensor, Bp: torch.Tensor, Br: torch.Tensor):
    device = next(model.parameters()).device
    X = X.to(device).float()
    tpts = tpts.to(device).float()
    Bp = Bp.to(device)
    Br = Br.to(device)
    Xhat, H, _ = model(X, tpts, Bp, Br)
    return Xhat.detach().cpu(), H.detach().cpu()


@torch.no_grad()
def fae_var_forecast(model: FAEVanilla, Xn_train: torch.Tensor, steps: int, tpts: torch.Tensor, Bp: torch.Tensor, Br: torch.Tensor, var_ridge: float):
    _, H_train = fae_reconstruct(model, Xn_train, tpts, Bp, Br)
    H_np = H_train.numpy()
    b, A = fit_var1(H_np, ridge=var_ridge)
    Hf = forecast_var1(H_np[-1], b, A, steps=steps)

    device = next(model.parameters()).device
    Hf_t = torch.tensor(Hf, dtype=torch.float32, device=device)
    X_fore = model.decode_from_rep(Hf_t, Br.to(device)).detach().cpu()
    return X_fore


# ============================================================
# 8) Run one simulation
# ============================================================

@dataclass
class RunCfg:
    horizon: int = 5
    fpca_K: int = 1
    fpca_var_ridge: float = 1e-6


def run_two_reports(sim_cfg: SimCfg, run_cfg: RunCfg, fae_cfg: FaeCfg) -> None:
    sim = simulate_functional_ts(sim_cfg)
    u = sim["u"]
    tpts = sim["tpts"]
    X_clean = sim["X_clean"]
    X_noisy = sim["X_noisy"]

    Xc_train, Xc_test = chrono_split(X_clean, run_cfg.horizon)
    Xn_train, Xn_test = chrono_split(X_noisy, run_cfg.horizon)

    print("Shapes:")
    print("  train:", tuple(Xn_train.shape), " test:", tuple(Xn_test.shape), " horizon:", run_cfg.horizon)

    X_fpca_recon_train, mu_fpca, phi_fpca, scores_train = fpca_reconstruct_from_trainfit(
        X_train=Xn_train, X_eval=Xn_train, u=u, K=run_cfg.fpca_K
    )

    model_fae, tpts_fae, Bp_fae, Br_fae = train_fae_on_train(Xn_train, Xc_train, tpts, fae_cfg)
    X_fae_recon_train, _ = fae_reconstruct(model_fae, Xn_train, tpts_fae, Bp_fae, Br_fae)

    recon_rows = {
        "FPCA": {
            "relMSE_vs_clean": rel_mse(X_fpca_recon_train, Xc_train),
            "relMSE_vs_noisy": rel_mse(X_fpca_recon_train, Xn_train),
        },
        "FAE": {
            "relMSE_vs_clean": rel_mse(X_fae_recon_train, Xc_train),
            "relMSE_vs_noisy": rel_mse(X_fae_recon_train, Xn_train),
        },
    }
    print_report_table("REPORT 1 — Reconstruction on TRAIN", recon_rows)

    X_fpca_fore = fpca_var_forecast(
        X_train=Xn_train, u=u, K=run_cfg.fpca_K, steps=run_cfg.horizon, var_ridge=run_cfg.fpca_var_ridge
    )

    X_fae_fore = fae_var_forecast(
        model=model_fae,
        Xn_train=Xn_train,
        steps=run_cfg.horizon,
        tpts=tpts_fae,
        Bp=Bp_fae,
        Br=Br_fae,
        var_ridge=fae_cfg.var_ridge,
    )

    fore_rows = {
        "FPCA+VAR": {
            "relMSE_vs_clean": rel_mse(X_fpca_fore, Xc_test),
            "relMSE_vs_noisy": rel_mse(X_fpca_fore, Xn_test),
        },
        "FAE+VAR": {
            "relMSE_vs_clean": rel_mse(X_fae_fore, Xc_test),
            "relMSE_vs_noisy": rel_mse(X_fae_fore, Xn_test),
        },
    }
    print_report_table(f"REPORT 2 — Forecast on TEST horizon (H={run_cfg.horizon})", fore_rows)


# ============================================================
# 9) Batch runner
# ============================================================

def _build_fixed_decoder(sim_cfg: SimCfg) -> Dict[str, object]:
    set_seed(sim_cfg.seed)

    if sim_cfg.map_mode.lower() == "nonlinear":
        u = np.linspace(sim_cfg.age_min, sim_cfg.age_max, sim_cfg.P).astype(float)
    else:
        u = np.linspace(0.0, 1.0, sim_cfg.P).astype(float)

    tpts = torch.tensor(u, dtype=torch.float32)
    Sigma = (sim_cfg.Sigma_scale ** 2) * np.eye(sim_cfg.d)

    if sim_cfg.map_mode.lower() == "linear":
        _, A_true = generate_latent_var1(
            T=sim_cfg.T, d=sim_cfg.d, Sigma=Sigma, seed=sim_cfg.seed, target_rho=sim_cfg.target_rho
        )

        gen_builder = BasisFCBuilder(
            n_basis=sim_cfg.gen_n_basis,
            basis_type=sim_cfg.gen_basis_type,
            bspline_degree=sim_cfg.gen_bspline_degree,
        )
        Bgen = gen_builder.build(tpts)

        torch.manual_seed(sim_cfg.seed)
        mapper = LinearMap(d=sim_cfg.d, hidden=[20], M=sim_cfg.gen_n_basis, bias=True)
        for m in mapper.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=sim_cfg.map_weight_sd)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        mapper.eval()
        for p in mapper.parameters():
            p.requires_grad_(False)

        return {
            "u": u,
            "tpts": tpts,
            "Sigma": Sigma,
            "A_true": A_true,
            "Bgen": Bgen,
            "mapper": mapper,
            "mode": "linear",
        }

    return {
        "u": u,
        "tpts": tpts,
        "Sigma": Sigma,
        "mode": "nonlinear",
    }


def _simulate_new_dataset_same_decoder(fixed: Dict[str, object], sim_cfg: SimCfg, data_seed: int):
    Sigma = fixed["Sigma"]
    T, d = sim_cfg.T, sim_cfg.d

    if fixed["mode"] == "linear":
        rng = np.random.default_rng(data_seed)
        A = fixed["A_true"]
        Z = np.zeros((T, d), dtype=float)
        Z[0] = rng.multivariate_normal(np.zeros(d), Sigma)
        for t in range(1, T):
            eps = rng.multivariate_normal(np.zeros(d), Sigma)
            Z[t] = A @ Z[t - 1] + eps
        Z_t = torch.tensor(Z, dtype=torch.float32)

        mapper: nn.Module = fixed["mapper"]
        Bgen: torch.Tensor = fixed["Bgen"]
        with torch.no_grad():
            coef = mapper(Z_t)
            X_clean = coef @ Bgen.T
            X_noisy = X_clean + sim_cfg.meas_noise_sd * torch.randn_like(X_clean)
    else:
        Z = generate_latent_phase_friendly(T=T, d=d, Sigma=Sigma, seed=data_seed)
        Z_t = torch.tensor(Z, dtype=torch.float32)
        with torch.no_grad():
            X_clean = phase_warped_fertility_decoder(Z_t, sim_cfg)
            X_noisy = X_clean + sim_cfg.meas_noise_sd * torch.randn_like(X_clean)

    if not torch.isfinite(X_clean).all() or not torch.isfinite(X_noisy).all():
        raise ValueError("Generated curves contain non-finite values.")

    return X_clean, X_noisy


def _run_one_rep_and_collect(
    fixed: Dict[str, object],
    sim_cfg: SimCfg,
    run_cfg: RunCfg,
    fae_cfg: FaeCfg,
    rep: int,
    data_seed_base: int,
) -> Dict[str, float]:
    data_seed = data_seed_base + rep

    X_clean, X_noisy = _simulate_new_dataset_same_decoder(fixed, sim_cfg, data_seed=data_seed)
    u = fixed["u"]
    tpts = fixed["tpts"]

    Xc_train, Xc_test = chrono_split(X_clean, run_cfg.horizon)
    Xn_train, Xn_test = chrono_split(X_noisy, run_cfg.horizon)

    X_fpca_recon_train, _, _, _ = fpca_reconstruct_from_trainfit(
        X_train=Xn_train, X_eval=Xn_train, u=u, K=run_cfg.fpca_K
    )

    model_fae, tpts_fae, Bp_fae, Br_fae = train_fae_on_train(Xn_train, Xc_train, tpts, fae_cfg)
    X_fae_recon_train, _ = fae_reconstruct(model_fae, Xn_train, tpts_fae, Bp_fae, Br_fae)

    n = Xn_train.shape[0]
    n_tr = max(8, int(fae_cfg.split_rate * n))
    n_tr = min(n_tr, n - 1)

    TrainData = Xn_train[:n_tr].float()
    ValData = Xn_train[n_tr:].float()
    CleanTrain = Xc_train[:n_tr].float()
    CleanVal = Xc_train[n_tr:].float()

    device = torch.device(fae_cfg.device)
    model_fae.eval()
    with torch.no_grad():
        tr_hat, _, _ = model_fae(TrainData.to(device), tpts_fae.to(device), Bp_fae.to(device), Br_fae.to(device))
        va_hat, _, _ = model_fae(ValData.to(device), tpts_fae.to(device), Bp_fae.to(device), Br_fae.to(device))

    fae_train_mse_final = mse(tr_hat.detach().cpu(), CleanTrain)
    fae_val_mse_final = mse(va_hat.detach().cpu(), CleanVal)

    fpca_recon_rel_clean = rel_mse(X_fpca_recon_train, Xc_train)
    fpca_recon_rel_noisy = rel_mse(X_fpca_recon_train, Xn_train)
    fae_recon_rel_clean = rel_mse(X_fae_recon_train, Xc_train)
    fae_recon_rel_noisy = rel_mse(X_fae_recon_train, Xn_train)

    X_fpca_fore = fpca_var_forecast(
        X_train=Xn_train, u=u, K=run_cfg.fpca_K, steps=run_cfg.horizon, var_ridge=run_cfg.fpca_var_ridge
    )
    X_fae_fore = fae_var_forecast(
        model=model_fae,
        Xn_train=Xn_train,
        steps=run_cfg.horizon,
        tpts=tpts_fae,
        Bp=Bp_fae,
        Br=Br_fae,
        var_ridge=fae_cfg.var_ridge,
    )

    fpca_fore_rel_clean = rel_mse(X_fpca_fore, Xc_test)
    fpca_fore_rel_noisy = rel_mse(X_fpca_fore, Xn_test)
    fae_fore_rel_clean = rel_mse(X_fae_fore, Xc_test)
    fae_fore_rel_noisy = rel_mse(X_fae_fore, Xn_test)

    return {
        "rep": rep,
        "data_seed": data_seed,

        "fpca_recon_relMSE_vs_clean": fpca_recon_rel_clean,
        "fpca_recon_relMSE_vs_noisy": fpca_recon_rel_noisy,
        "fae_recon_relMSE_vs_clean": fae_recon_rel_clean,
        "fae_recon_relMSE_vs_noisy": fae_recon_rel_noisy,

        "fae_train_mse_final_clean": fae_train_mse_final,
        "fae_val_mse_final_clean": fae_val_mse_final,

        "fpca_fore_relMSE_vs_clean": fpca_fore_rel_clean,
        "fpca_fore_relMSE_vs_noisy": fpca_fore_rel_noisy,
        "fae_fore_relMSE_vs_clean": fae_fore_rel_clean,
        "fae_fore_relMSE_vs_noisy": fae_fore_rel_noisy,
    }


def run_sim_fpca_fae_many_to_excel(
    sim_cfg: SimCfg,
    run_cfg: RunCfg,
    fae_cfg: FaeCfg,
    n_reps: int = 20,
    out_xlsx_path: str = "sim_results.xlsx",
    data_seed_base: int = 20000,
    verbose: bool = False,
) -> pd.DataFrame:
    fixed = _build_fixed_decoder(sim_cfg)

    rows: List[Dict[str, float]] = []
    for rep in range(1, n_reps + 1):
        if not verbose:
            old_log = fae_cfg.log_every
            fae_cfg.log_every = 0
        try:
            row = _run_one_rep_and_collect(
                fixed=fixed,
                sim_cfg=sim_cfg,
                run_cfg=run_cfg,
                fae_cfg=fae_cfg,
                rep=rep,
                data_seed_base=data_seed_base,
            )
        finally:
            if not verbose:
                fae_cfg.log_every = old_log

        row.update({
            "map_mode": sim_cfg.map_mode,
            "T": sim_cfg.T,
            "P": sim_cfg.P,
            "d": sim_cfg.d,
            "meas_noise_sd": sim_cfg.meas_noise_sd,
            "fpca_K": run_cfg.fpca_K,
            "horizon": run_cfg.horizon,
            "fae_n_rep": fae_cfg.n_rep,
            "fae_n_basis_project": fae_cfg.n_basis_project,
            "fae_n_basis_revert": fae_cfg.n_basis_revert,
        })
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_excel(out_xlsx_path, index=False)
    return df


# ============================================================
# 10) Example
# ============================================================

if __name__ == "__main__":
    # Nonlinear scenario intentionally favorable to FAE
    sim_cfg = SimCfg(
        seed=123,
        T=500,
        P=100,
        d=5,
        Sigma_scale=0.028,
        target_rho=0.85,
        gen_basis_type="Bspline",
        gen_n_basis=12,
        gen_bspline_degree=3,
        map_mode="nonlinear",
        map_weight_sd=0.8,
        meas_noise_sd=0.0010,
        age_min=15.0,
        age_max=49.0,
        fertility_tail_strength=0.012,
        fertility_floor=0.0,
        fertility_scale=1.0,
    )

    # keep FPCA low-rank in the nonlinear phase-warp case
    run_cfg = RunCfg(
        horizon=5,
        fpca_K=3,
        fpca_var_ridge=1e-6,
    )

    fae_cfg = FaeCfg(
        seed=743,
        device="cpu",
        n_basis_project=45,
        n_basis_revert=45,
        basis_type_project="Bspline",
        basis_type_revert="Bspline",
        bspline_degree=3,
        n_rep=3,
        init_weight_sd=0.06,
        epochs=1400,
        batch_size=16,
        lr=7e-4,
        weight_decay=1e-5,
        split_rate=0.85,
        log_every=200,
        smooth_lambda=2e-4,
        dyn_lambda=8.0,
        var_ridge=1e-6,
    )

    run_two_reports(sim_cfg, run_cfg, fae_cfg)

    df = run_sim_fpca_fae_many_to_excel(
        sim_cfg=sim_cfg,
        run_cfg=run_cfg,
        fae_cfg=fae_cfg,
        n_reps=10,
        out_xlsx_path="sim_results_fae_friendly.xlsx",
        data_seed_base=30000,
        verbose=True,
    )

    print(df.head())

Shapes:
  train: (495, 100)  test: (5, 100)  horizon: 5
[FAE] ep  200 | train_mse(clean_target)=8.805878e-05 | val_mse(clean_target)=9.747616e-05
[FAE] ep  400 | train_mse(clean_target)=9.776958e-05 | val_mse(clean_target)=1.387236e-04
[FAE] ep  600 | train_mse(clean_target)=9.440103e-05 | val_mse(clean_target)=9.776375e-05
[FAE] ep  800 | train_mse(clean_target)=8.254703e-05 | val_mse(clean_target)=1.211182e-04
[FAE] ep 1000 | train_mse(clean_target)=6.614096e-05 | val_mse(clean_target)=1.068779e-04
[FAE] ep 1200 | train_mse(clean_target)=4.906577e-05 | val_mse(clean_target)=1.028568e-04
[FAE] ep 1400 | train_mse(clean_target)=2.074021e-05 | val_mse(clean_target)=9.717848e-05

REPORT 1 — Reconstruction on TRAIN
Method             relMSE vs CLEAN   relMSE vs NOISY
----------------------------------------------------
FPCA                      0.119773          0.120454
FAE                       0.086565          0.087353

REPORT 2 — Forecast on TEST horizon (H=5)
Method             relM